# calc_U_h_coeffs.ipynb

This script implements heat transfer correlations based in boundary layers to estimate heat transfer coefficients for natural convection at the tank bottom and tank sidewalls. Correlations for the Nusselt number over different interfaces are presented as a function of the characteristic length, $l_0$, of convection:

- $h_b$: Heat transfer coefficient by Natural Convection at the tank bottom, using $l_o = d_o$ where $l_o$ is the characteristic length.
- $h_i$: Heat transfer coefficient by Natural Convection between the liquid ammonia and the internal tank wall, using $l_o = H$.
- $h_{o,nc}$: Heat transfer coefficient by Natural Convection between the air and the external tank wall, using $l_o = H$.
- $h_{o,fc}$: Heat transfer coefficient by Forced Convection between the air and the external tank wall, using $l_o = \pi d_o /(2H+2d_o)$.

Source: 


In [1]:
# Import modules
import CoolProp.CoolProp as CP
import numpy as np

In [57]:
# Raw data from https://es.weatherspark.com/y/26544/Clima-promedio-en-Diego-de-Almagro-Chile-durante-todo-el-a%C3%B1o#:~:text=En%20Diego%20de%20Almagro%2C%20los,m%C3%A1s%20de%2029%20%C2%B0C.
T_month = [21, 21, 20, 19, 17, 15, 14, 16, 17, 18, 19, 20]

tavg = np.mean(T_month)
print(f'Average temperature = {tavg:.2f} °C')

wind_velocity = [12.7, 11.0, 9.2, 10.9] #km/h
print(f'Wind velocity = {np.mean(wind_velocity)*(1000/3600):.4f} m/s')

Average temperature = 18.08 °C
Wind velocity = 3.0417 m/s


In [ ]:
# CONSTANT PARAMETERS
fluid    = 'Air'  
Air      = 'Air' 
pressure = 101325 * 3  # Operating pressure 3 atm in Pa

# LNG tank properties
Q_roof = 0              # Roof heat ingress / W
T_air  = 18.08+273.15  # Temperature of the environment K
V_tank = 4831           # Tank volume / m^3
a      = 0.5            # aspect ratio H/D
d_i    = ((4 * V_tank)/(np.pi * a))**(1/3) # internal diameter / m
e      = 10.6972*2 - 10.24*2 # twice the wall thickness / m
d_o    = d_i + e       # External diameter / m
H      = a*d_i         # Height in meters
g      = 9.81          # Acceleration due to gravity in m/s^2
T_air  = 18.08 + 273.15 # Ambient air temperature / K, Diego de Almagro, Atacama, Chile
P_air  = 101325        # / Pa
LF     = 0.95          # Liquid fraction in the tank

# Tank properties
e_p = 18*0.0254 # Perlite insulation layer thickness / m
k_p = 0.0411    # Perlite thermal conductivity Wm^-1K^-1

# Stainless steel dopado con Mn
e_m = 1*0.0254 # Metalic layer insulation layer thickness / m
k_m = 32.5     # Metalic layer thermal conductivity Wm^-1K^-1

# Start of iteration
delta_T = 0.1  # Temperature difference in K

# Get the saturation temperature at 1 atm
T_sat = CP.PropsSI('T', 'P', pressure, 'Q', 0, fluid)# + 1e-3

# We assess thermophysical properties at film temperature
T_f = T_sat + delta_T/2

# Assumption: LAES is not boiling at the tank bottom
# so the properties of the liquid are calculated
# assuming saturation AT THE FILM TEMPERATURE
rho     = CP.PropsSI('D', 'Q', 0, 'T', T_f, fluid)
rho_air = CP.PropsSI('D','P',P_air,'T', T_air, Air)

# Get the derivative of density with respect to temperature at constant pressure
# drho_dT = CP.PropsSI('d(D)/d(T)|P', 'P', pressure, 'Q', 0, fluid)
drho_dT = CP.PropsSI('d(D)/d(T)|P', 'T', T_f, 'Q', 0, fluid)
drho_dT_air = CP.PropsSI('d(D)/d(T)|P', 'T', T_air, 'P', P_air, Air)

# Calculate the thermal expansion coefficient
beta = - drho_dT / rho
beta_air = - drho_dT_air/rho_air

# Get the Prandtl number at saturation temperature and 1 atm
Pr     = CP.PropsSI('Prandtl', 'T', T_f, 'Q', 0, fluid)
Pr_air = CP.PropsSI('Prandtl', 'T', T_air, 'P', P_air, Air) 

# Get the dynamic viscosity at saturation temperature and 1 atm
mu     = CP.PropsSI('V', 'T', T_f, 'Q', 0, fluid)
mu_air = CP.PropsSI('V','T', T_air, 'P', P_air, Air)

# Calculate the kinematic viscosity
nu     = mu / rho
nu_air = mu_air/rho_air

# Get the thermal conductivity at saturation temperature and 1 atm
k     = CP.PropsSI('L', 'T', T_f, 'Q', 0, fluid)
k_air = CP.PropsSI('L','T', T_air, 'P', P_air, Air)

# Calculate the thermal diffusivity
alpha     = k / (rho * CP.PropsSI('C', 'T', T_f, 'Q', 0, fluid))
alpha_air = k_air/(rho_air * CP.PropsSI('C', 'T', T_air, 'P', P_air, Air))
# Calculate the Rayleigh number
Ra     = (g * beta * delta_T * H**3) / (nu * alpha)
Ra_air = (g * beta_air * delta_T * H**3) / (nu_air * alpha_air)

print(f'Thermal expansion coefficient of liquid ammonia at saturation temperature and 1 atm: {beta:.6e} 1/K')
print(f'Prandtl number of ammonia: {Pr:.6f}')
print(f'Rayleigh number of ammonia: {Ra:.6e}')
print(f'Thermal expansion coefficient of air at atmospheric conditions: {beta_air:.6e} 1/K')
print(f'Prandtl number of air: {Pr_air:.6f}')
print(f'Rayleigh number or air: {Ra_air:.6e}')

Thermal expansion coefficient of liquid ammonia at saturation temperature and 1 atm: 6.241885e-03 1/K
Prandtl number of ammonia: 1.919730
Rayleigh number of ammonia: 9.030055e+14
Thermal expansion coefficient of air at atmospheric conditions: 3.443741e-03 1/K
Prandtl number of air: 0.708215
Rayleigh number or air: 1.648937e+10


## Functions definition

In [78]:
# Calculates internal heat transfer coefficient at the base of a cylindrical tank
def h_i_base(Pr, Ra, k, D):
    f1 = (1 + (0.492*Pr)**(9/16))**(-16/9)
    print(f'Raf1: {Ra*f1:.6e}')
    # Heat emission at lower surface 
    Nu_b = 0.6 * (Ra * f1)**(1/5)
    print(f'Nusselt number: {Nu_b:.3e}')
    h_b = Nu_b * k / D 
    print(f'Internal heat transfer coefficient at the tank base, h_b: {h_b:.3e}', 'Wm^-2K^-1')
    return h_b

def U_base(h_b, r_i, r_o, k_p):
    '''
    This equation does not include the heat transfer from the outside
    Inputs:
        h_b: heat transfer coefficient from the bottom
        r_i: tank internal radius / m
        r_2: metallic layer radius / m
        k_m: metallic layer thermal conductivity / Wm^-1K^-1
        r_o: insulation olayer radius / m
        k_p: insulation layer thermal conductivity / Wm^-1K^-1

    The resistance of the inner metallic wall is neglected
    '''
    U = (r_o/(r_i*h_b) + r_o * np.log(r_o/r_i)/k_p)**(-1)
    return U

def h_i_sides(Pr, Ra, k, H, D):
    f1 = (1+(0.492/Pr)**(9/16))**(-16/9)
    Nu_plate = (0.825+0.387*(Ra*f1)**(1/6))**2
    Nu = Nu_plate + 0.97*(H/D)
    print(f'Ra number: {Ra:.3e}')
    print(f'Nusselt number: {Nu:.3e}')

    # The characteristic length is the height of the tank for vertical
    # natural convection
    h_sides = Nu * k / H 

    print(f'Internal heat transfer coefficient at the tank sides, h_side: {h_sides:.3e}', 'Wm^-2K^-1')
    return h_sides

def h_forced_sides(Pr, Re, k, d):
    Nu_lam = 0.664 * Re**(1/2) * Pr**(1/3)
    Nu_turb = (0.037 * Re**(0.8) * Pr)/(1 + 2.443 * Re**(-0.1) * (Pr**(2/3) - 1))
    Nu = 0.3 + (Nu_lam**2 + Nu_turb**2)**(1/2)

    # Streamed length
    # l_0 = np.pi * d / 2
    l_0 = (np.pi * d_o * H) / (2 * (H + d_o)) 
    h_forced = Nu * k / l_0
    print(f'External heat transfer coefficient at the tank sides, h_forced_sides: {h_forced:.3e}', 'Wm^-2K^-1')
    return h_forced

def U_sides(h_si, h_so, r_i, r_o, k_p):
    '''

    Overall heat transfer coefficient at the tank sides
    
    Inputs:
        h_si: internal heat transfer coefficient
        h_so: external heat transfer coefficient
        r_i: tank internal radius / m
        r_2: metallic layer radius / m
        k_m: metallic layer thermal conductivity / Wm^-1K^-1
        r_o: insulation layer radius / m
        k_p: insulation layer thermal conductivity / Wm^-1K^-1
    '''
    U = ( r_o/(r_i*h_si) + r_o * (np.log(r_o/r_i)/k_p ) + 1/(h_so))**(-1)
    return U


# Alternative correlation
def h_cylinder_crossflow(U, D, T_inf, T_s, P=101325.0):
    # film temperature
    T_f = 0.5*(T_inf + T_s)
    # air properties at film T and P
    rho = CP.PropsSI('D','P',P,'T',T_f,'Air')
    mu  = CP.PropsSI('VISCOSITY','P',P,'T',T_f,'Air')
    k   = CP.PropsSI('CONDUCTIVITY','P',P,'T',T_f,'Air')
    cp  = CP.PropsSI('CPMASS','P',P,'T',T_f,'Air')
    Pr  = cp*mu/k
    Re  = rho*U*D/mu

    # Churchill–Bernstein (valid for all Re, Pr)
    term = (0.62*Re**0.5*Pr**(1/3)) / (1 + (0.4/Pr)**(2/3))**0.25
    Nu   = 0.3 + term * (1 + (Re/282000.0)**(5/8))**(4/5)
    return Nu * k / D

## Local Heat Transfer Coefficients

In [64]:
h_so = h_i_sides(Pr_air, Ra_air, k_air, H, d_o) # External heat transfer coefficient

h_si = h_i_sides(Pr, Ra, k_air, LF*H, d_i) # Internal heat transfer coefficient

Ra number: 1.649e+10
Nusselt number: 2.959e+02
Internal heat transfer coefficient at the tank sides, h_side: 6.597e-01 Wm^-2K^-1
Ra number: 9.030e+14
Nusselt number: 1.172e+04
Internal heat transfer coefficient at the tank sides, h_side: 2.751e+01 Wm^-2K^-1


$$ h(T_w - T_L) = U(T_{air} - T_L)$$

In [65]:
# Internal radius
r_i = d_i/2

# Radius to the end of the insulation layer
r_o = r_i + e_p

# Assumption: temperature of the outer wall is the same as the air temperature
h_b = h_i_base(Pr, Ra, k, d_i)

# Based on external area
U_b = U_base(h_b, r_i, r_o, k_p)

Raf1: 2.709092e+14
Nusselt number: 4.621e+02
Internal heat transfer coefficient at the tank base, h_b: 2.429e+00 Wm^-2K^-1


The correlation works for $$ 10^3 < Ra*f_1 \leq 10^{10} $$

## Automated iteration for the bottom of the tank

In [66]:
err_dT = 1 # K
tol = 1e-4 # err
delta_T = 0.01 # K

while err_dT > tol:
    # Get the saturation temperature at 1 atm
    T_sat = CP.PropsSI('T', 'P', pressure, 'Q', 0, fluid)# + 1e-3

    # We assess thermophysical properties at film temperature
    T_f = T_sat + delta_T/2

    # Assumption: LAES is not boiling at the tank bottom
    # so the properties of the liquid are calculated
    # assuming saturation AT THE FILM TEMPERATURE
    rho = CP.PropsSI('D', 'Q', 0, 'T', T_f, fluid)

    # Get the derivative of density with respect to temperature at constant pressure
    drho_dT = CP.PropsSI('d(D)/d(T)|P', 'T', T_f, 'Q', 0, fluid)

    # Calculate the thermal expansion coefficient
    beta = - drho_dT / rho

    # Get the Prandtl number at saturation temperature and 1 atm
    Pr = CP.PropsSI('Prandtl', 'T', T_f, 'Q', 0, fluid)

    # Get the dynamic viscosity at saturation temperature and 1 atm
    mu = CP.PropsSI('V', 'T', T_f, 'Q', 0, fluid)

    # Calculate the kinematic viscosity
    nu = mu / rho

    # Get the thermal conductivity at saturation temperature and 1 atm
    k = CP.PropsSI('L', 'T', T_f, 'Q', 0, fluid)

    # Calculate the thermal diffusivity
    alpha = k / (rho * CP.PropsSI('C', 'T', T_f, 'Q', 0, fluid))

    # Calculate the Rayleigh number
    Ra = (g * beta * delta_T * H**3) / (nu * alpha)
    
    # Assumption: temperature of the outer wall is the same as the air temperature
    h_b = h_i_base(Pr, Ra, k, d_i)

    # Based on external area
    U_b = U_base(h_b, r_i, r_o, k_p)
    
    delta_T_new = U_b/h_b * (T_air-T_sat)
    err_dT = abs(delta_T_new - delta_T)
    print("dT_new = %.3f K " % delta_T_new)
    print("dT_err", err_dT)
    delta_T = delta_T_new

Raf1: 2.701764e+13
Nusselt number: 2.914e+02
Internal heat transfer coefficient at the tank base, h_b: 1.533e+00 Wm^-2K^-1
dT_new = 10.924 K 
dT_err 10.914326658464415
Raf1: 4.112032e+16
Nusselt number: 1.262e+03
Internal heat transfer coefficient at the tank base, h_b: 6.131e+00 Wm^-2K^-1
dT_new = 2.852 K 
dT_err 8.072578486278347
Raf1: 8.393687e+15
Nusselt number: 9.182e+02
Internal heat transfer coefficient at the tank base, h_b: 4.734e+00 Wm^-2K^-1
dT_new = 3.677 K 
dT_err 0.8254093407677909
Raf1: 1.109642e+16
Nusselt number: 9.709e+02
Internal heat transfer coefficient at the tank base, h_b: 4.977e+00 Wm^-2K^-1
dT_new = 3.501 K 
dT_err 0.17588441913741582
Raf1: 1.050964e+16
Nusselt number: 9.604e+02
Internal heat transfer coefficient at the tank base, h_b: 4.929e+00 Wm^-2K^-1
dT_new = 3.534 K 
dT_err 0.033184389204682585
Raf1: 1.061989e+16
Nusselt number: 9.624e+02
Internal heat transfer coefficient at the tank base, h_b: 4.938e+00 Wm^-2K^-1
dT_new = 3.528 K 
dT_err 0.006413533875

In [67]:
print("U_b = %.5e W m^-2 K^-1 " % U_b)

U_b = 8.65640e-02 W m^-2 K^-1 


## Automated iteration for the sides of the tank

Liquid side

In [68]:
err_dT = 1
tol = 1e-8
delta_T_side_i = 1
h_so = h_i_sides(Pr_air, Ra_air, k_air, H, d_o) # External heat transfer coefficient

# Liquid phase iteration

while err_dT > tol:
    # Get the saturation temperature at 1 atm
    T_sat = CP.PropsSI('T', 'P', pressure, 'Q', 0, fluid)# + 1e-3

    # Evaluate thermophysical properties at film temperature
    T_f = T_sat + delta_T_side_i/2

    # Assumption: LAES is boiling at the tank bottom
    # so the properties of the liquid are calculated
    # assuming saturation AT THE FILM TEMPERATURE
    rho = CP.PropsSI('D', 'Q', 0, 'T', T_f, fluid)

    # Get the derivative of density with respect to temperature at constant pressure
    drho_dT = CP.PropsSI('d(D)/d(T)|P', 'T', T_f, 'Q', 0, fluid)

    # Calculate the thermal expansion coefficient
    beta = - drho_dT / rho

    # Get the Prandtl number at saturation temperature and 1 atm
    Pr = CP.PropsSI('Prandtl', 'T', T_f, 'Q', 0, fluid)

    # Get the dynamic viscosity at saturation temperature and 1 atm
    mu = CP.PropsSI('V', 'T', T_f, 'Q', 0, fluid)

    # Calculate the kinematic viscosity
    nu = mu / rho

    # Get the thermal conductivity at saturation temperature and 1 atm
    k = CP.PropsSI('L', 'T', T_f, 'Q', 0, fluid)

    # Calculate the thermal diffusivity
    alpha = k / (rho * CP.PropsSI('C', 'T', T_f, 'Q', 0, fluid))

    # Calculate the Rayleigh number
    Ra = (g * beta * delta_T_side_i * H**3) / (nu * alpha)

#    print(f'Thermal expansion coefficient of liquid ammonia at saturation temperature and 1 atm: {beta:.6e} 1/K')
#    print(f'Prandtl number: {Pr:.6f}')
#    print(f'Rayleigh number: {Ra:.6e}')
    
    # Assumption: temperature of the outer wall is the same as the air temperature
    h_si = h_i_sides(Pr, Ra, k, LF*H, d_i) # Internal heat transfer coefficient

    # Based on external area
    U_s = U_sides(h_si, h_so, r_i, r_o, k_p)
    
    delta_T_new = U_s/(h_si * d_o/d_i) * (T_air-T_sat)
    err_dT = abs(delta_T_new - delta_T_side_i)
    print("dT_err", err_dT)
    delta_T_side_i = delta_T_new



Ra number: 1.649e+10
Nusselt number: 2.959e+02
Internal heat transfer coefficient at the tank sides, h_side: 6.597e-01 Wm^-2K^-1
Ra number: 9.255e+15
Nusselt number: 2.533e+04
Internal heat transfer coefficient at the tank sides, h_side: 2.785e+02 Wm^-2K^-1
dT_err 0.9459564385638979
Ra number: 4.874e+14
Nusselt number: 9.561e+03
Internal heat transfer coefficient at the tank sides, h_side: 1.058e+02 Wm^-2K^-1
dT_err 0.08810159611660223
Ra number: 1.285e+15
Nusselt number: 1.317e+04
Internal heat transfer coefficient at the tank sides, h_side: 1.458e+02 Wm^-2K^-1
dT_err 0.03890046712372701
Ra number: 9.324e+14
Nusselt number: 1.185e+04
Internal heat transfer coefficient at the tank sides, h_side: 1.311e+02 Wm^-2K^-1
dT_err 0.011520573468081274
Ra number: 1.037e+15
Nusselt number: 1.227e+04
Internal heat transfer coefficient at the tank sides, h_side: 1.358e+02 Wm^-2K^-1
dT_err 0.003946398039201923
Ra number: 1.001e+15
Nusselt number: 1.213e+04
Internal heat transfer coefficient at the t

Air

In [69]:
err_dT = 1
tol = 1e-8
delta_T_side_o = 0.1
# Air  iteration

while err_dT > tol:
    # Evaluate thermophysical properties at film temperature
    T_f = T_air + delta_T_side_o/2

    # Air density
    rho_air = CP.PropsSI('D','P', P_air,'T', T_f, Air)

    # Get the derivative of density with respect to temperature at constant pressure
    drho_dT = CP.PropsSI('d(D)/d(T)|P', 'P',P_air, 'T', T_f, Air)

    # Calculate the thermal expansion coefficient
    beta_air = - drho_dT / rho_air

    # Get the Prandtl number at saturation temperature and 1 atm
    Pr_air = CP.PropsSI('Prandtl', 'P',P_air, 'T', T_f, Air)

    # Get the dynamic viscosity at saturation temperature and 1 atm
    mu_air = CP.PropsSI('V', 'P',P_air, 'T', T_f, Air)

    # Calculate the kinematic viscosity
    nu_air = mu_air / rho_air

    # Get the thermal conductivity at saturation temperature and 1 atm
    k_air = CP.PropsSI('L', 'P',P_air, 'T', T_f, Air)

    # Calculate the thermal diffusivity
    alpha_air = k_air / (rho_air * CP.PropsSI('C', 'P',P_air, 'T', T_f, Air))

    # Calculate the Rayleigh number
    Ra_air = (g * beta_air * delta_T_side_o * H**3) / (nu_air * alpha_air)
    print("Ra_air = %.3e" % Ra_air)
    
    # Calculate new external heat transfer coefficient
    h_so = h_i_sides(Pr_air, Ra_air, k_air, H, d_o) # Internal heat transfer coefficient

    # Based on external area
    U_s = U_sides(h_si, h_so, r_i, r_o, k_p)
    
    delta_T_new = U_s/(h_so) * (T_air-T_sat)
    err_dT = abs(delta_T_new - delta_T_side_o)
    print("dT_err", err_dT)
    delta_T_side_o = delta_T_new



Ra_air = 1.648e+10
Ra number: 1.648e+10
Nusselt number: 2.959e+02
Internal heat transfer coefficient at the tank sides, h_side: 6.596e-01 Wm^-2K^-1
dT_err 23.616194202655695
Ra_air = 3.251e+12
Ra number: 3.251e+12
Nusselt number: 1.624e+03
Internal heat transfer coefficient at the tank sides, h_side: 3.745e+00 Wm^-2K^-1
dT_err 19.090029293468454
Ra_air = 7.353e+11
Ra number: 7.353e+11
Nusselt number: 1.002e+03
Internal heat transfer coefficient at the tank sides, h_side: 2.248e+00 Wm^-2K^-1
dT_err 2.9658787213065727
Ra_air = 1.179e+12
Ra number: 1.179e+12
Nusselt number: 1.167e+03
Internal heat transfer coefficient at the tank sides, h_side: 2.631e+00 Wm^-2K^-1
dT_err 1.0708144343109343
Ra_air = 1.021e+12
Ra number: 1.021e+12
Nusselt number: 1.114e+03
Internal heat transfer coefficient at the tank sides, h_side: 2.507e+00 Wm^-2K^-1
dT_err 0.31114474867551856
Ra_air = 1.067e+12
Ra number: 1.067e+12
Nusselt number: 1.130e+03
Internal heat transfer coefficient at the tank sides, h_side: 2

In [70]:
print(U_s) # Global Heat Transfer coefficient from the sides of the tank
print(U_b) # Global Heat Transfer coefficient from the bottom

print(f'Global heat transfer coefficient at the tank base, U_b: {U_b:.3e}', 'Wm^-2K^-1')
print(f'Global heat transfer coefficient at the tank sides, U_s: {U_s:.3e}', 'Wm^-2K^-1')

0.08515270410165926
0.08656404079730405
Global heat transfer coefficient at the tank base, U_b: 8.656e-02 Wm^-2K^-1
Global heat transfer coefficient at the tank sides, U_s: 8.515e-02 Wm^-2K^-1


In [71]:
print(h_so)
print(h_b)
print(h_si)

2.5359160065880224
4.936642876770148
134.61081154153217


Forced convection

In [88]:
print(f'Ra_air_ext = {Ra_air:.6e}')
print(f'Ra_air_int = {Ra:.6e}')

Ra_air_ext = 1.648937e+10
Ra_air_int = 9.030055e+14


In [89]:
v_air = 3.041 # m s^-1

# Characteristic length is the streamwise length
l_0 = (np.pi * d_o * H) / (2 * (H + d_o)) 

Re = rho_air * v_air * l_0 / mu_air
h_forced = h_forced_sides(Pr_air, Re, k_air, d_o)

U_FC = U_sides(h_si, h_forced, r_i, r_o, k_p)
print("h_so_fc = %.4e W m^-2 K ^-1 " % h_forced)
print("U_FC| VDI Heat Atlas  = %.5e W m^-2 K ^-1 " % U_FC)

h_so_fcx = h_cylinder_crossflow(v_air, d_o, T_air, T_air, P_air)
U_FC = U_sides(h_si, h_so_fcx, r_i, r_o, k_p)
print("h_so_fc = %.4e W m^-2 K ^-1 " % h_so_fcx)
print("U_FC | cross flow = %.4e W m^-2 K ^-1 " % U_FC)

External heat transfer coefficient at the tank sides, h_forced_sides: 8.387e+00 Wm^-2K^-1
h_so_fc = 8.3869e+00 W m^-2 K ^-1 
U_FC| VDI Heat Atlas  = 8.71953e-02 W m^-2 K ^-1 
h_so_fc = 5.4201e+00 W m^-2 K ^-1 
U_FC | cross flow = 8.6702e-02 W m^-2 K ^-1 
